# Uber Fare Prediction — Model Training & Comparison

Train and compare five regression approaches on the preprocessed dataset: **Linear Regression**, **Multiple Regression** (OLS with full statistical summary), **Ridge Regression**, **Lasso Regression**, and **XGBoost**. Each model is evaluated with MAE, RMSE, and R², with a final side-by-side comparison. The notebook then runs 5-fold cross-validation and predicts the fare for a sample 15 km trip using every model.

## 1. Import Libraries

Core libraries for data handling, train/test splitting, the five regression models, and evaluation metrics.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor

pd.set_option('display.max_columns', None)

## 2. Load Data

Load the preprocessed, fully-encoded dataset (output of the data analysis/preprocessing notebook).

In [ ]:
data = pd.read_csv('processed_data.csv')
print("Shape:", data.shape)
data.head()

## 3. Basic Checks

Confirm the dataset is model-ready: no missing values, and all columns are numeric.

In [ ]:
print("Missing values:", data.isnull().sum().sum())
print("\nData types:\n", data.dtypes.value_counts())

## 4. Feature / Target Split & Train-Test Split

Separate features (`X`) from the target (`Fare (Target)`), then split into training (80%) and test (20%) sets. All five models below are trained and evaluated on this same split for a fair comparison.

In [ ]:
X = data.drop(columns=['Fare (Target)'])
y = data['Fare (Target)']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

## 5. Evaluation Helper

A small reusable function to compute and print MAE, RMSE, and R² for any model's predictions, and store the results for the final comparison.

In [ ]:
results = {}

def evaluate_model(name, y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    results[name] = {'MAE': mae, 'RMSE': rmse, 'R2': r2}
    print(f"{name}")
    print(f"  MAE : {mae:.2f}")
    print(f"  RMSE: {rmse:.2f}")
    print(f"  R2  : {r2:.4f}")

## 6. Linear Regression

A baseline ordinary least squares (OLS) model — fits one straight-line relationship between the features and Fare, assuming no interaction between features and constant error variance.

In [ ]:
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

lr_pred = lr_model.predict(X_test)
evaluate_model('Linear Regression', y_test, lr_pred)

## 7. Multiple Regression (Detailed OLS Summary)

`LinearRegression` above is already technically a *multiple* regression, since it fits on many features at once — but here we use `statsmodels` to get the full statistical picture: coefficients, p-values, and confidence intervals for each feature, which sklearn doesn't provide directly. This tells us not just *how well* the model predicts, but *which features are statistically significant* contributors to Fare.

In [ ]:
# statsmodels needs numeric (float) input -- cast first (boolean dummy columns cause errors otherwise)
X_train_sm = sm.add_constant(X_train.astype(float))
X_test_sm = sm.add_constant(X_test.astype(float))

mr_model = sm.OLS(y_train, X_train_sm).fit()
print(mr_model.summary())

In [ ]:
mr_pred = mr_model.predict(X_test_sm)
evaluate_model('Multiple Regression (OLS)', y_test, mr_pred)

## 8. Ridge Regression (L2 Regularization)

Ridge adds a penalty on large coefficients, which helps when features are correlated with each other (e.g. Distance and Duration) — it shrinks coefficients rather than eliminating them, making the model more stable without fully discarding any feature.

In [ ]:
ridge_model = Ridge(alpha=1.0, random_state=42)
ridge_model.fit(X_train, y_train)

ridge_pred = ridge_model.predict(X_test)
evaluate_model('Ridge Regression', y_test, ridge_pred)

## 9. Lasso Regression (L1 Regularization)

Lasso also penalizes large coefficients, but can shrink weak features' coefficients all the way to zero — effectively performing automatic feature selection. Useful for spotting which features the model considers unnecessary.

In [ ]:
lasso_model = Lasso(alpha=1.0, random_state=42, max_iter=5000)
lasso_model.fit(X_train, y_train)

lasso_pred = lasso_model.predict(X_test)
evaluate_model('Lasso Regression', y_test, lasso_pred)

### Features Lasso eliminated (coefficient shrunk to zero)

In [ ]:
lasso_coefs = pd.Series(lasso_model.coef_, index=X_train.columns)
eliminated = lasso_coefs[lasso_coefs == 0]
print(f"Lasso eliminated {len(eliminated)} of {len(lasso_coefs)} features:")
print(eliminated.index.tolist())

## 10. XGBoost Regression

XGBoost (Extreme Gradient Boosting) is a powerful tree-based ensemble model that often outperforms linear models on tabular data. It builds trees sequentially, each correcting the errors of the previous one, while regularisation parameters (`subsample`, `colsample_bytree`) help prevent overfitting. It's trained here — before the comparison below — so it's included in the same side-by-side results as the other four models.

In [ ]:
xgb_model = XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)
xgb_model.fit(X_train, y_train)

xgb_pred = xgb_model.predict(X_test)
evaluate_model('XGBoost', y_test, xgb_pred)

## 11. Model Comparison

All five models — Linear Regression, Multiple Regression (OLS), Ridge, Lasso, and XGBoost — evaluated on the identical test set, side by side.

In [ ]:
comparison_df = pd.DataFrame(results).T
comparison_df = comparison_df.sort_values(by='R2', ascending=False)
comparison_df

### Visual comparison of R², MAE, and RMSE across models

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

comparison_df['R2'].plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('R² Score by Model')
axes[0].set_ylabel('R²')
axes[0].tick_params(axis='x', rotation=45)

comparison_df['MAE'].plot(kind='bar', ax=axes[1], color='darkorange')
axes[1].set_title('MAE by Model')
axes[1].set_ylabel('MAE')
axes[1].tick_params(axis='x', rotation=45)

comparison_df['RMSE'].plot(kind='bar', ax=axes[2], color='indianred')
axes[2].set_title('RMSE by Model')
axes[2].set_ylabel('RMSE')
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

### Summary

- **Linear Regression** and **Multiple Regression (OLS)** produce nearly identical metrics — expected, since they're the same underlying method; the OLS summary above additionally shows which individual features are statistically significant.
- **Ridge** typically performs very close to Linear Regression here, since its main benefit (stabilizing correlated features) only shows a large effect when multicollinearity is severe.
- **Lasso** may perform slightly worse if it zeroed out features that actually carried real signal — check the eliminated feature list above to see if that happened.
- **XGBoost** is expected to outperform the four linear-family models if the relationship between features and Fare is not purely linear, since it can capture non-linear patterns and feature interactions — check its position in the table/plots above to confirm.

## 12. Cross-Validation (5-Fold R²)

5-fold cross-validation on the training set gives a more robust estimate of generalisation performance by averaging results across five different train/validation splits, reducing the risk of a lucky (or unlucky) single split.

In [ ]:
cv_scores = cross_val_score(xgb_model, X_train, y_train, cv=5, scoring='r2')
print("CV R² scores:", cv_scores)
print("Mean CV R²:", cv_scores.mean())

## 13. Sample Fare Prediction — Bike vs Sedan (15 km), All Models

Construct a single synthetic trip (15 km, pleasant weather, low traffic, midday, residential areas, normal day) and predict its fare for both a **Bike** and a **Sedan** using every trained model — Linear Regression, Multiple Regression (OLS), Ridge, Lasso, and XGBoost — so their real-world predictions can be compared directly.

In [ ]:
# Create a row with all features initialised as 0, matching X_train's columns exactly
bike_prediction = pd.DataFrame(0, index=[0], columns=X_train.columns)

# Numeric features
bike_prediction['Distance (km)'] = 15
bike_prediction['Estimated Duration (min)'] = 30

# Normal conditions
bike_prediction['Weather Condition'] = 0        # Clear
bike_prediction['Rainfall Intensity'] = 0       # No Rain
bike_prediction['Traffic Level'] = 0            # Low
bike_prediction['Temperature Level'] = 1        # Pleasant
bike_prediction['Holiday'] = 0                  # No Holiday
bike_prediction['Day Type'] = 0                 # Weekday
bike_prediction['Number of Stops Added'] = 0

# Normal categorical values
bike_prediction['Pickup Area Type_Residential'] = 1
bike_prediction['Drop Area Type_Residential'] = 1
bike_prediction['Time Slot_Midday'] = 1
bike_prediction['Busy Day_Normal'] = 1

# Vehicle type = Bike
bike_prediction['Vehicle Type_Bike'] = 1

# Sedan version of the same trip
sedan_prediction = bike_prediction.copy()
sedan_prediction['Vehicle Type_Bike'] = 0
sedan_prediction['Vehicle Type_Sedan'] = 1

# statsmodels (Multiple Regression) needs the constant column added, same as during training
bike_prediction_sm = sm.add_constant(bike_prediction.astype(float), has_constant='add')
sedan_prediction_sm = sm.add_constant(sedan_prediction.astype(float), has_constant='add')

# Predict with every trained model
sample_predictions = {
    'Linear Regression': (
        lr_model.predict(bike_prediction)[0],
        lr_model.predict(sedan_prediction)[0]
    ),
    'Multiple Regression (OLS)': (
        mr_model.predict(bike_prediction_sm)[0],
        mr_model.predict(sedan_prediction_sm)[0]
    ),
    'Ridge Regression': (
        ridge_model.predict(bike_prediction)[0],
        ridge_model.predict(sedan_prediction)[0]
    ),
    'Lasso Regression': (
        lasso_model.predict(bike_prediction)[0],
        lasso_model.predict(sedan_prediction)[0]
    ),
    'XGBoost': (
        xgb_model.predict(bike_prediction)[0],
        xgb_model.predict(sedan_prediction)[0]
    ),
}

sample_df = pd.DataFrame(sample_predictions, index=['Predicted Bike Fare (₹)', 'Predicted Sedan Fare (₹)']).T
sample_df = sample_df.round(2)
sample_df